# Week 4 — Practical Lab: A Complete Data Workflow

So far each tool has been learned on its own. This lab puts them together in the order a real analyst uses them.

**The dataset.** `students.csv` — records for a university's students: their program, city, study habits, attendance, previous GPA and final marks. It is deliberately *messy*, the way real data always is.

**The workflow.** Every data project follows roughly the same six stages, and this lab walks through all of them:

| Stage | What it means | Tools |
|---|---|---|
| 1. Load | Get the data into memory | `read_csv` |
| 2. Inspect | Understand its size, columns and types | `head`, `info`, `describe` |
| 3. Clean | Fix duplicates, inconsistencies, missing values | `drop_duplicates`, `fillna` |
| 4. Organize | Filter, sort, and create useful new columns | masks, `cut`, `apply` |
| 5. Analyze | Answer questions about groups and relationships | `groupby`, `merge`, `corr` |
| 6. Report | State findings in plain language | markdown |

**A note on the order.** It is not arbitrary. You cannot trust an average computed from duplicated rows, and you cannot compute a correlation on a column full of gaps. Cleaning comes before analysis because analysis done on dirty data produces confident, wrong answers.

In [59]:
import numpy as np
import pandas as pd

---
# Stage 1 — Load the data

`read_csv` reads a comma-separated file into a DataFrame. Both files should sit in the same folder as this notebook.

In [60]:
df = pd.read_csv(r"C:\artificial_intelligence_coarse\sir_git_ai\Artificial-Inteligence-Machine-Learning-and-Deep-Learning-NAVTTC-Course-2026\week-04-libraries-descriptive-statistics\datasets\students.csv")
city_info = pd.read_csv(r"C:\artificial_intelligence_coarse\sir_git_ai\Artificial-Inteligence-Machine-Learning-and-Deep-Learning-NAVTTC-Course-2026\week-04-libraries-descriptive-statistics\datasets\city_info.csv")

print("students :", df.shape)      # (rows, columns)
print("city_info:", city_info.shape)

students : (184, 10)
city_info: (6, 3)


---
# Stage 2 — Inspect

Before changing anything, look at what you have. Three questions: *how big is it, what are the columns, and what is missing?*

In [61]:
df.head(5)

,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
1,S1179,Female,Karachi,BSSE,7,5.2,77.4,NaN,54.4,0
2,S1131,Female,Lahore,BBA,7,4.0,96.9,NaN,68.7,0
3,S1024,Female,PESHAWAR,BSCS,3,2.5,100.0,2.84,56.8,0
4,S1065,Male,Quetta,BSSE,4,7.7,87.8,NaN,68.8,0


**What this does:** shows the first 5 rows. It is the fastest way to see whether the file loaded correctly, whether the column names came through, and what the values look like.

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   student_id   184 non-null    str    
 1   gender       184 non-null    str    
 2   city         184 non-null    str    
 3   program      184 non-null    str    
 4   semester     184 non-null    int64  
 5   study_hours  179 non-null    float64
 6   attendance   168 non-null    float64
 7   prev_gpa     174 non-null    float64
 8   final_marks  184 non-null    float64
 9   scholarship  184 non-null    int64  
dtypes: float64(4), int64(2), str(4)
memory usage: 14.5 KB


**What this does:** lists every column with its data type and its count of non-missing values. Any column whose count is below the total row count has gaps. Note `study_hours`, `attendance`, and `prev_gpa` here.

In [63]:
df.describe().round(2)

,semester,study_hours,attendance,prev_gpa,final_marks,scholarship
count,184.00,179.00,168.00,174.00,184.00,184.00
mean,4.78,7.26,82.16,2.84,65.12,0.42
std,2.29,4.89,10.50,0.58,11.51,0.50
min,1.00,0.50,54.20,1.37,32.10,0.00
25%,3.00,3.65,76.00,2.46,58.18,0.00
50%,5.00,6.40,82.70,2.88,65.70,0.00
75%,7.00,8.85,89.40,3.26,72.72,1.00
max,8.00,30.00,100.00,4.00,92.40,1.00


**What this does:** the five-number summary plus mean and standard deviation, for every numeric column. Scan it for impossible values — a negative age, an attendance above 100, a GPA above 4.0 would all signal a data-entry problem.

Note the gap between the **mean** and the **50%** (median) of `study_hours`: when the mean sits noticeably above the median, the column is **right-skewed** — a few students study far more than the rest.

In [64]:
df.isnull().sum()

student_id      0
gender          0
city            0
program         0
semester        0
study_hours     5
attendance     16
prev_gpa       10
final_marks     0
scholarship     0
dtype: int64

In [65]:
# Which columns are missing, as a percentage
(df.isnull().sum() / len(df) * 100).round(1)

student_id     0.0
gender         0.0
city           0.0
program        0.0
semester       0.0
study_hours    2.7
attendance     8.7
prev_gpa       5.4
final_marks    0.0
scholarship    0.0
dtype: float64

**What this does:** quantifies the gaps. Under about 5% is usually easy to fill; a column missing 40%+ may be worth dropping entirely. Here the gaps are small enough to fill.

---
# Stage 3 — Clean

Three problems to fix, in order: duplicated rows, inconsistent text, and missing values.

In [66]:
df

,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
1,S1179,Female,Karachi,BSSE,7,5.2,77.4,NaN,54.4,0
2,S1131,Female,Lahore,BBA,7,4.0,96.9,NaN,68.7,0
3,S1024,Female,PESHAWAR,BSCS,3,2.5,100.0,2.84,56.8,0
4,S1065,Male,Quetta,BSSE,4,7.7,87.8,NaN,68.8,0
...,...,...,...,...,...,...,...,...,...,...
179,S1103,Female,Peshawar,BSCS,4,5.5,68.6,3.73,56.0,1
180,S1151,Female,Peshawar,BSSE,6,3.5,70.1,2.37,63.5,0
181,S1067,Male,Lahore,BSSE,7,1.4,82.0,2.14,65.3,0
182,S1025,Male,Peshawar,BSEE,2,7.9,97.8,2.35,71.5,0


### 3.1 Duplicate rows

A duplicate is the same record entered twice. It silently biases every average, because one student gets counted twice.

In [67]:
df[df.duplicated(keep=False)]

,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
13,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
76,S1161,Female,Multan,BSCS,7,6.4,56.8,2.71,53.4,1
91,S1116,Female,Multan,BBA,6,4.9,83.3,2.87,61.2,0
105,S1116,Female,Multan,BBA,6,4.9,83.3,2.87,61.2,0
135,S1112,Male,Multan,BSCS,2,13.8,84.9,3.45,74.1,1
150,S1161,Female,Multan,BSCS,7,6.4,56.8,2.71,53.4,1
157,S1112,Male,Multan,BSCS,2,13.8,84.9,3.45,74.1,1


In [68]:
print("duplicate rows:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("student_id").head(8)

duplicate rows: 4


,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
13,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0
135,S1112,Male,Multan,BSCS,2,13.8,84.9,3.45,74.1,1
157,S1112,Male,Multan,BSCS,2,13.8,84.9,3.45,74.1,1
91,S1116,Female,Multan,BBA,6,4.9,83.3,2.87,61.2,0
105,S1116,Female,Multan,BBA,6,4.9,83.3,2.87,61.2,0
76,S1161,Female,Multan,BSCS,7,6.4,56.8,2.71,53.4,1
150,S1161,Female,Multan,BSCS,7,6.4,56.8,2.71,53.4,1


In [69]:
df = df.drop_duplicates()
print("rows after removing duplicates:", len(df))

rows after removing duplicates: 180


**What this does:** `duplicated()` flags rows identical to an earlier row; `drop_duplicates()` removes them, keeping the first occurrence.

### 3.2 Inconsistent categories

Text entered by hand is rarely consistent. To the computer, `"Male"`, `"male"` and `"MALE"` are three different categories.

In [70]:
print(df["gender"].unique())
print(df["city"].unique())
#df['city']=df['city'].apply(lambda s :s.title())

<StringArray>
['Female', 'Male', 'male', 'female']
Length: 4, dtype: str
<StringArray>
[   'Quetta',   'Karachi',    'Lahore',  'PESHAWAR',  'Peshawar',   'KARACHI',
    'Multan', 'Islamabad']
Length: 8, dtype: str


In [71]:
# Standardize: strip stray spaces, then apply consistent capitalization
df["gender"] = df["gender"].str.strip().str.title()
df["city"]   = df["city"].str.strip().str.title()

print(df["gender"].unique())
print(df["city"].unique())

<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
['Quetta', 'Karachi', 'Lahore', 'Peshawar', 'Multan', 'Islamabad']
Length: 6, dtype: str


**What this does:** `.str.title()` converts each value to Title Case, so `"male"` and `"MALE"` both become `"Male"`. Without this step, a `groupby("gender")` would produce four groups instead of two, and every summary would be wrong.

### 3.3 Missing values

Now decide, column by column, what each gap should become.

In [72]:
for col in ["study_hours", "attendance", "prev_gpa"]:
    print(f"{col:12s}    mean= {df[col].mean():6.2f}   median={df[col].median():6.2f}   skew={df[col].skew():5.2f}")

study_hours     mean=   7.23   median=  6.40   skew= 1.50
attendance      mean=  82.25   median= 82.65   skew=-0.32
prev_gpa        mean=   2.84   median=  2.88   skew=-0.19


In [73]:
# Skewed column → median.  Roughly symmetric column → mean is acceptable, median is still safe.
df["study_hours"] = df["study_hours"].fillna(df["study_hours"].median())
df["attendance"]  = df["attendance"].fillna(df["attendance"].median())
df["prev_gpa"]    = df["prev_gpa"].fillna(df["prev_gpa"].median())

print("remaining missing values:", df.isnull().sum().sum())

remaining missing values: 0


**What this does, and why the median.** `study_hours` is right-skewed — a handful of very high values drag the mean upward, so the mean is not a typical student. The median is unaffected by those extremes, which makes it the safer substitute.

**The honest caveat:** filling gaps is a decision, not a fact. Every filled cell is a guess. It pulls values toward the centre and slightly *reduces* the real spread of the column, so a standard deviation computed after filling is a little smaller than the truth. That is an acceptable trade for keeping the rows — but it must be a conscious choice, and it should be written down.

---
# Stage 4 — Organize

The data is now trustworthy. Next: filter it, sort it, and add columns that make the analysis easier.

In [74]:
# Filtering with a boolean mask — high achievers
high = df[df["final_marks"] > 80]
print(f"{len(high)} students scored above 80 ({len(high)/len(df)*100:.1f}% of the class)")

high[['study_hours','attendance','prev_gpa','final_marks']].head(16)

16 students scored above 80 (8.9% of the class)


,study_hours,attendance,prev_gpa,final_marks
8,30.0,78.70,3.23,92.4
31,27.1,63.60,2.70,85.8
35,6.1,100.00,2.57,85.0
56,9.1,89.60,2.43,83.9
67,6.4,100.00,2.95,80.6
75,15.8,69.50,2.96,84.8
77,19.3,77.10,2.84,80.9
88,14.1,82.65,3.10,83.0
89,18.5,63.50,3.26,85.5
96,7.9,99.30,3.45,82.6


In [75]:
# Multiple conditions — each wrapped in its own parentheses
focused = df[(df["study_hours"] > 10) & (df["attendance"] > 85)]
print("students who study a lot AND attend regularly:", len(focused))
focused[['student_id',"study_hours",'prev_gpa','attendance','final_marks']]

students who study a lot AND attend regularly: 11


,student_id,study_hours,prev_gpa,attendance,final_marks
40,S1011,15.0,2.12,87.3,71.2
72,S1041,10.8,2.82,86.5,78.0
83,S1079,10.4,2.58,88.5,74.5
110,S1124,11.6,2.69,91.5,72.8
115,S1165,11.8,2.88,98.6,72.1
131,S1133,10.3,1.71,89.4,59.5
136,S1100,14.4,2.81,99.3,86.0
143,S1071,18.9,3.63,90.1,79.1
154,S1038,10.9,2.73,90.0,64.4
169,S1068,12.2,1.90,93.8,74.4


In [76]:
# Sorting — top 5 performers
df.sort_values("final_marks", ascending=False).head(5)[
    ["student_id", "program", "city", "study_hours", "final_marks"]
]

,student_id,program,city,study_hours,final_marks
8,S1121,BSCS,Peshawar,30.0,92.4
174,S1089,BBA,Peshawar,18.9,91.8
149,S1083,BSCS,Peshawar,18.2,90.4
146,S1118,BBA,Islamabad,19.9,88.4
136,S1100,BBA,Peshawar,14.4,86.0


### Creating new columns

Two ways to derive a new column: **binning** a continuous value into bands, and **applying** a function to compute one.

In [77]:
# Binning: continuous marks → an ordered grade category
df["grade"] = pd.cut(
    df["final_marks"],
    bins=[0, 50, 60, 70, 80, 100],
    labels=["F", "D", "C", "B", "A"]
)
df[["final_marks", "grade"]].head()

,final_marks,grade
0,68.5,C
1,54.4,D
2,68.7,C
3,56.8,D
4,68.8,C


In [78]:
df.head()

,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship,grade
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0,C
1,S1179,Female,Karachi,BSSE,7,5.2,77.4,2.88,54.4,0,D
2,S1131,Female,Lahore,BBA,7,4.0,96.9,2.88,68.7,0,C
3,S1024,Female,Peshawar,BSCS,3,2.5,100.0,2.84,56.8,0,D
4,S1065,Male,Quetta,BSSE,4,7.7,87.8,2.88,68.8,0,C


In [79]:
df["grade"].value_counts().sort_index()

grade
F    23
D    30
C    63
B    48
A    16
Name: count, dtype: int64

In [80]:
df["grade"].value_counts()

grade
C    63
B    48
D    30
F    23
A    16
Name: count, dtype: int64

**What this does:** `pd.cut` assigns each mark to a band. It converts a *quantitative continuous* column into a *qualitative ordinal* one — the grades have a natural order (A > B > C). This is the conversion from the data-types lesson, applied for real.

In [81]:
# Applying a function to compute a new column
df["study_level"] = df["study_hours"].apply(
    lambda h: "high" if h >= 10 else ("medium" if h >= 5 else "low")
)
df[["study_hours", "study_level"]].head()

,study_hours,study_level
0,8.1,medium
1,5.2,medium
2,4.0,low
3,2.5,low
4,7.7,medium


---
# Stage 5 — Analyze

Now the actual questions. Each one is a single line of Pandas.

### 5.1 Group comparisons

**Question: does the amount of studying show up in the marks?**

In [82]:
df.groupby("study_level", observed=True)["final_marks"].mean().round(2)

study_level
high      76.04
low       57.46
medium    66.17
Name: final_marks, dtype: float64

In [83]:
df["final_marks"].mean().round(2)

np.float64(65.14)

In [84]:
# The fuller picture: how many students, and how much variation, in each group
df.groupby("study_level", observed=True)["final_marks"].agg( ["count", "mean", "std", "min", "max"]).round(2)

,count,mean,std,min,max
study_level,,,,,
high,35,76.04,9.87,44.5,92.4
low,61,57.46,10.25,32.1,76.6
medium,84,66.17,8.88,42.5,85.0


**What this does:** splits students by study level, then computes several statistics for each group. The `count` column matters — a group's mean is unreliable if only a few students are in it. The `std` shows how consistent each group is: a high standard deviation means students in that band vary widely despite similar study habits.

In [85]:
# Question: how does performance differ across programs?
df.groupby("program")["final_marks"].agg(["count", "mean", "std"]).round(2).sort_values(by="mean", ascending=False)

,count,mean,std
program,,,
BBA,43,67.74,12.24
BSCS,72,64.93,11.62
BSSE,40,64.54,11.43
BSEE,25,62.22,10.22


In [86]:
# Grouping by two columns at once
df.groupby(["program", "gender"])["final_marks"].mean().round(2)

program  gender
BBA      Female    66.78
         Male      68.58
BSCS     Female    65.21
         Male      64.77
BSEE     Female    64.55
         Male      60.06
BSSE     Female    60.29
         Male      67.08
Name: final_marks, dtype: float64

In [87]:
# Scholarship rate by program — the mean of a 0/1 column is a proportion
(df.groupby("program")["scholarship"].mean() * 100).round(1)

program
BBA     37.2
BSCS    38.9
BSEE    52.0
BSSE    47.5
Name: scholarship, dtype: float64

**What this does:** because `scholarship` holds only 0 and 1, its mean is the fraction of 1s — the scholarship rate. Multiplying by 100 turns it into a percentage. This trick works for any yes/no column stored as 0/1.

### 5.2 Merging in the lookup table

The student records hold a city name but nothing about that city. A second table holds the province and campus type. Merging attaches that information to every student.

In [88]:
city_info

,city,province,campus_type
0,Peshawar,Khyber Pakhtunkhwa,Main
1,Lahore,Punjab,Main
2,Karachi,Sindh,Main
3,Islamabad,Islamabad Capital Territory,Satellite
4,Quetta,Balochistan,Satellite
5,Multan,Punjab,Satellite


In [89]:
df = pd.merge(df, city_info, on="city",how='left')
df[["student_id", "city", "province", "campus_type"]].head()

,student_id,city,province,campus_type
0,S1099,Quetta,Balochistan,Satellite
1,S1179,Karachi,Sindh,Main
2,S1131,Lahore,Punjab,Main
3,S1024,Peshawar,Khyber Pakhtunkhwa,Main
4,S1065,Quetta,Balochistan,Satellite


**What this does, and why `left`.** A left join keeps **every** student row and attaches city information where it matches. Had we used `inner`, any student from a city missing from the lookup table would silently vanish from the analysis. When you have a main table and are adding reference information to it, `left` is almost always the right choice.

Always verify a merge did what you expected:

In [90]:
print("rows after merge:", len(df))                       # should be unchanged
print("unmatched cities:", df['province'].isnull().sum())  # should be 0

rows after merge: 180
unmatched cities: 0


In [97]:
df

,student_id,gender,city,program,semester,study_hours,attendance,prev_gpa,final_marks,scholarship,grade,study_level,province,campus_type
0,S1099,Female,Quetta,BBA,1,8.1,87.5,1.91,68.5,0,C,medium,Balochistan,Satellite
1,S1179,Female,Karachi,BSSE,7,5.2,77.4,2.88,54.4,0,D,medium,Sindh,Main
2,S1131,Female,Lahore,BBA,7,4.0,96.9,2.88,68.7,0,C,low,Punjab,Main
3,S1024,Female,Peshawar,BSCS,3,2.5,100.0,2.84,56.8,0,D,low,Khyber Pakhtunkhwa,Main
4,S1065,Male,Quetta,BSSE,4,7.7,87.8,2.88,68.8,0,C,medium,Balochistan,Satellite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,S1103,Female,Peshawar,BSCS,4,5.5,68.6,3.73,56.0,1,D,medium,Khyber Pakhtunkhwa,Main
176,S1151,Female,Peshawar,BSSE,6,3.5,70.1,2.37,63.5,0,C,low,Khyber Pakhtunkhwa,Main
177,S1067,Male,Lahore,BSSE,7,1.4,82.0,2.14,65.3,0,C,low,Punjab,Main
178,S1025,Male,Peshawar,BSEE,2,7.9,97.8,2.35,71.5,0,B,medium,Khyber Pakhtunkhwa,Main


In [91]:
# Now a question we could not previously ask
df.groupby("province")["final_marks"].agg(["count", "mean"]).round(2).sort_values("mean", ascending=False)

,count,mean
province,,
Balochistan,21,67.97
Islamabad Capital Territory,20,67.18
Khyber Pakhtunkhwa,58,66.68
Punjab,45,62.90
Sindh,36,62.66


In [92]:
df.groupby("campus_type")["final_marks"].mean().round(2)

campus_type
Main         64.77
Satellite    65.93
Name: final_marks, dtype: float64

### 5.3 Correlation — which factors track final marks?

In [ ]:
numeric = ["study_hours", "attendance", "prev_gpa", "final_marks"]
df[numeric].corr().round(3)

,study_hours,attendance,prev_gpa,final_marks
study_hours,1.000,-0.033,0.056,0.655
attendance,-0.033,1.000,0.009,0.300
prev_gpa,0.056,0.009,1.000,0.280
final_marks,0.655,0.300,0.280,1.000


In [102]:
df[["study_hours", "attendance", "prev_gpa", "final_marks"]].corr().round(1)

,study_hours,attendance,prev_gpa,final_marks
study_hours,1.0,-0.0,0.1,0.7
attendance,-0.0,1.0,0.0,0.3
prev_gpa,0.1,0.0,1.0,0.3
final_marks,0.7,0.3,0.3,1.0


In [103]:
# Just the column we care about, sorted
df[numeric].corr()["final_marks"].drop("final_marks").sort_values(ascending=False).round(3)

study_hours    0.655
attendance     0.300
prev_gpa       0.280
Name: final_marks, dtype: float64

**How to read this.** Each number is between −1 and +1 and measures how strongly that column moves together with `final_marks`. Study hours shows the strongest relationship, attendance a moderate one, previous GPA a weaker one.

**And the essential caution:** a strong correlation does **not** prove that studying *causes* higher marks. Motivated students may both study more *and* attend more *and* score higher — motivation would be the hidden cause behind all three. Correlation identifies a pattern worth investigating; it never settles the question of cause.

In [95]:
# Correlations can also be computed within a group
df.groupby("program", observed=True)[["study_hours", "final_marks"]].corr().round(3).iloc[0::2, 1]

program             
BBA      study_hours    0.753
BSCS     study_hours    0.628
BSEE     study_hours    0.722
BSSE     study_hours    0.570
Name: final_marks, dtype: float64

---
# Stage 6 — Report

An analysis nobody can read has no value. Close every project by stating what you found in plain sentences, with the numbers that support each claim.

Run the cell below to gather the key figures, then write your findings underneath.

In [119]:
print("FINAL DATASET")
print(f"  students: {len(df)}   columns: {df.shape[1]}   missing values: {df.isnull().sum().sum()}")
print()
print("MARKS")
print(f"  mean : {df['final_marks'].mean():.1f}   median : {df['final_marks'].median():.1f}   std : {df['final_marks'].std():.1f}")
print()
print("BY STUDY LEVEL")
print(df.groupby("study_level", observed=True)["final_marks"].mean().round(1).to_string())
print()
print("STRONGEST DRIVER OF MARKS")
c = df[numeric].corr()["final_marks"].drop("final_marks")
print(f"  {c.idxmax()} (r = {c.max():.3f})")

FINAL DATASET
  students: 180   columns: 14   missing values: 0

MARKS
  mean : 65.1   median : 65.7   std : 11.6

BY STUDY LEVEL
study_level
high      76.0
low       57.5
medium    66.2

STRONGEST DRIVER OF MARKS
  study_hours (r = 0.655)


### Your findings

*(Write 3–5 bullet points here in plain English. Each should state something you found and the number that backs it up. Example: "Students in the 'high' study group averaged X marks, compared with Y for the 'low' group — a difference of Z.")*

1. student in the high study group avg marks were 76 that due to .apply function making it easy to apply a function inside pandas on study hour in then connecting them with each 
2. (.apply) function were first applied first on study_hours to amke another column in dataframe  
3. (.groupby) function were used to group the data based on any group with corresponding another group

---
## What this workflow accomplished

Starting from a messy file, we:

- **removed** duplicated records that would have double-counted students,
- **standardized** inconsistent category text so grouping produced correct counts,
- **filled** missing values with the median, keeping every row while recording the trade-off,
- **derived** grade bands and study levels that made comparison possible,
- **grouped** to compare programs, genders, provinces and study habits,
- **merged** external reference data to unlock questions the original file could not answer,
- **measured** which factors track final marks — while refusing to claim causation.

Every one of these steps used a tool from this week. The tools are not the point; **the order is.** Clean before you analyze, verify after you merge, and always state your findings in words a non-programmer could follow.